<h1> <a id="title"></a>Differencing and geostatistical error analysis in lidar topographic differencing: Workflow starting with user provided point clouds</h1>

This work was funded by the [John Wesley Powell Center for Analysis and Synthesis (USGS G23AC00336)](https://www.usgs.gov/centers/john-wesley-powell-center-for-analysis-and-synthesis/science/a-national-topographic-change#overview) and [OpenTopography](https://opentopography.org).  OpenTopography is supported by the National Science Foundation under Awards # 2410799, 2410800 & 2410801.

<h2><a id="introduction"></a>1. Introduction</h2>

Vertical topographic differencing calculates net change in the vertical dimension by comparing digital elevation models (DEMs) collected at different times ([Izumida et al., 2017](https://doi.org/10.5194/nhess-17-1505-2017); [Wheaton et al., 2010](https://doi.org/10.1002/esp.1886)). Topographic differencing underpins a wide range of studies, spanning vegetation biomass changes, lava-flow emplacement ([Albino et al., 2015](https://doi.org/10.1002/2015JB011988)), fluvial and coastal floods ([Izumida et al., 2017](https://doi.org/10.5194/nhess-17-1505-2017)), landslides ([Lucieer et al., 2014](https://doi.org/10.1177/0309133313515293)), fine-scale fluvial sediment budgets ([Wheaton et al., 2010](https://doi.org/10.1002/esp.1886)), and tectonic activity ([Langridge et al., 2014](https://doi.org/10.1016/j.geomorph.2014.08.007); [Scott et al., 2018](https://doi.org/10.1029/2018JB015581)).

<h3><a id="differencing_def"></a>Vertical topographic differencing</h3>

Vertical differencing is the pixel-by-pixel subtraction of two digital-elevation models (DEMs) that share a common coordinate system and identical grid geometry (e.g., [Scott et al., 2021](https://doi.org/10.1130/GES02259.1)). Because every cell occupies the same planimetric position in both rasters, subtracting them yields a raster of elevation change ($\Delta z$). Researchers rarely inspect the differencing value of every cell individually. Instead, aggregate change is summarized over a specific feature or landform (net sediment deposition on a bar, net tree growth in a reforested stand, net inflation along a volcanic flank, etc.) The mean elevation change over a polygon $\Omega$ is:

$$\Delta z^{aggregate} = \frac{1}{N} \sum_{i=1}^{N} \Delta z_i$$

where the sum of $i$ individual topographic change measurements ($\Delta z_i$) is taken over the $N$ cells that fall inside $\Omega$.

<h3><a id="uncertainty_matters"></a>Why uncertainty matters</h3> 

The elevation change observed in a differenced raster can be decomposed as:

$$\Delta_{z}^{measured} = \Delta_{z}^{actual} + \Delta_{z}^{error}$$

where:
- **Δz<sub>actual</sub>** is the true vertical change (erosion, deposition, uplift, subsidence, construction, vegetation change, etc.)
- **Δz<sub>error</sub>** is the cumulative vertical error from data collection, processing, and alignment (GNSS/INS errors, boresight errors, DEM interpolation artifacts, alignment errors, metadata errors)

Errors present in either dataset are often of similar magnitude to the true change, particularly for legacy datasets acquired with less advanced instrumentation, lower point density, or incomplete metadata ([Glennie et al., 2014](https://doi.org/10.1002/2014GL059919)).


<h3><a id="error_types"></a>Error types in lidar topographic differencing</h3>

<h4><a id="short_scale"></a>Short-scale errors (meters to tens of meters)</h4>

| Error Type | Description | Key References |
|------------|-------------|----------------|
| **Random noise** | Sensor sensitivity, environmental conditions, and GNSS/IMU interference produce scattered positive/negative differences | [Glennie, 2007](https://doi.org/10.1515/jag.2007.017) |
| **Point misclassification** | Incorrect ground/vegetation/building classification creates meter-to-decameter artifacts with diffuse boundaries | [Passalacqua et al., 2015](https://doi.org/10.1016/j.earscirev.2015.05.012) |
| **Geometric distortion** | High laser incidence angles on steep slopes spread pulse energy, degrading measurement quality | [Schaer et al., 2007](https://doi.org/10.1117/12.717277) |


<h4><a id="mid_scale"></a>Mid-scale errors (hundred-meter scale)</h4> 

| Error Type | Description | Key References |
|------------|-------------|----------------|
| **Horizontal alignment errors** | Georeferencing offsets produce apparent vertical change correlated with topographic aspect; correctable via ICP registration | [Glennie et al., 2014](https://doi.org/10.1002/2014GL059919); [Besl & McKay, 1992](https://doi.org/10.1117/12.57955) |
| **Flight line striping** | Kinematic GNSS atmospheric delays create linear bands (hundreds of meters to kilometers wide) perpendicular to flight path | [Shan et al., 2007](https://doi.org/10.1201/9781420051438); [DeLong et al., 2022](https://doi.org/10.1029/2022EA002420) |

<h4><a id="long_scale"></a>Long-scale errors (kilometer scale)</h4>

| Error Type | Description | Key References |
|------------|-------------|----------------|
| **Instrument calibration biases** | Range calibration, uniform GNSS/IMU misalignment, or atmospheric corrections affect entire dataset uniformly | [Glennie, 2007](https://doi.org/10.1515/jag.2007.017); [Habib et al., 2009](https://doi.org/10.14358/PERS.75.10.1159) |
| **Geoid model errors** | Wrong geoid model produces 10–20 cm vertical shifts across the scene | Brigham et al. |
| **Ellipsoidal/orthometric confusion** | Mixing height systems causes vertical errors of tens of meters | Brigham et al. |


<h3><a id="workflow"></a>Our workflow</h3>

This notebook implements a geostatistical approach to quantify uncertainty in topographic differencing. The key insight is that **uncertainty is often structured, scale-dependent, and dominated by mid- and long-range correlation**.

**Notebook steps:**

1. [**Setup**](#setup): Install dependencies and configure the environment
2. [**Data Loading & Preprocessing**](#data-access): Import user-provided point clouds, extract and verify/update CRS metadata
3. [**CRS Transformation**](#CRS_transformation): Align coordinate reference systems, vertical datums, and epochs
4. [**Point Cloud Alignment**](#alignment): ICP co-registration to correct horizontal offsets between surveys
5. [**2D DEM Differencing**](#differencing): Generate DEMs from point clouds and compute pixel-by-pixel elevation change
6. [**Visualization**](#visualization): Plot DEMs, hillshades, slopes, and the difference raster
7. [**Stable Area Identification**](#define_stable_areas): Delineate control zones where no real change is expected
8. [**Descriptive Statistics**](#descriptive_stats): Characterize the error distribution and check stationarity assumptions
9. [**Systematic Error Estimation**](#estimate-error): Estimate and remove vertical bias using the median of stable-area differences
10. [**Variography**](#Variography): Fit nested variogram models to capture multi-scale spatial error structure
11. [**Uncertainty Propagation**](#uncertainty_propagation): Propagate the error model to features of interest via Monte Carlo integration

<h2><a id="setup"></a>2. Setup</h2>

<h3><a id="colab"></a>Running the notebook in Colab</h3>

For ease-of-use, it is suggested to launch and execute these notebooks on <a href="https://colab.research.google.com/">Google Colaboratory</a> (Colab, for short), Google's Cloud Platform. Dependencies will be installed on a virtual machine on Google's cloud servers and the code will be executed directly in your browser. A major benefit of this is that you will have direct access to Google's high-end CPU/GPUs and will not have to install any dependencies locally. All deliverables will be saved to your personal Google Drive. To experiment and run one of the below Jupyter Notebooks on Google Colab click the "Open in Colab" badge below.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Cassandra-Brigham/topographic-difference-uncertainty/blob/main/differencing_workflow_user_pointclouds.ipynb)

In [ ]:
import os, sys, pathlib

# --- Colab guard ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # 1) Mount Drive (idempotent)
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # 2) Check if condacolab is already fully configured
    # We verify both that conda exists AND that LD_LIBRARY_PATH is set
    conda_ready = (
        os.path.exists("/usr/local/bin/conda") and
        "LD_LIBRARY_PATH" in os.environ
    )
    
    if conda_ready:
        import condacolab
        condacolab.check()
        print("âœ“ Condacolab already installed and configured")
    else:
        print("Installing condacolab (kernel will restart)...")
        !pip install -q condacolab
        import condacolab
        condacolab.install()  # This restarts the kernel
else:
    print("Not running in Colab; skipping condacolab setup.")

**Kernel Restart Required**

If this is your first time running this notebook, the cell above will have installed `condacolab` and **automatically restarted the kernel**. This is expected behavior.

**Please proceed by running the cell below** to continue with the environment setup. The next cell will install PDAL and the remaining dependencies.

<h3><a id="params"></a>Your parameters</h3>

Set your data path here!

- **Google Colab users**: Update `DATA_PATH` in the cell below to point to your data folder on Google Drive (e.g., `/content/drive/MyDrive/lidar-project`). Your Drive will be mounted when you run the setup cells.
- **Local users**: Update `DATA_PATH` in the cell below to point to your local data folder (e.g., `/Users/yourname/Documents/lidar-project`).

In [ ]:
# Update your data path here

DATA_PATH = "your/path/here"
COMPARE_PC_NAME = "compare.laz"
REFERENCE_PC_NAME = "reference.laz"

# Set base data directory based on environment
from pathlib import Path
if IN_COLAB:
    BASE_DATA_DIR = Path(DATA_PATH)
    
    print(f"Using Colab data directory: {BASE_DATA_DIR}")
else:
    BASE_DATA_DIR = Path(DATA_PATH)
    
    print(f"Using local data directory: {BASE_DATA_DIR}")

# Create base directory if it doesn't exist
BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Helper function to resolve data paths
def get_data_path(*path_parts):
    return str(BASE_DATA_DIR / Path(*path_parts))

Sometimes the point cloud file will lack important metadata or the metadata will be parsed incorrectly. To remedy this, we can manually update the point cloud metadata if we know more information. Fill out this information to the best of your ability. This information can be found in several places:

- **OpenTopography data pages**: Each dataset has a metadata page with survey details including CRS, datum, survey date, and geoid information. Look under the "Overview" or "Coordinates and Classification" sections. The "Full Metadata" link in the "Overview" section will take you to the original data index file.
- **Survey reports**: Most lidar acquisitions include a project report (often a PDF) documenting collection parameters, coordinate systems, and vertical datums.
- **USGS metadata**: For USGS 3DEP and other national elevation data, metadata is available at the [USGS National Geospatial Technical Operations Center](https://thor-f5.er.usgs.gov/ngtoc/metadata/waf/elevation/)
- **NOAA Digital Coast**: For coastal lidar data, metadata can be found in the [NOAA Digital Coast portal](https://coast.noaa.gov/digitalcoast/) or the individual dataset index files, e.g., [NOAA NOS Coastal Lidar index](https://noaa-nos-coastal-lidar-pds.s3.amazonaws.com/laz/geoid18/8866/index.html)
- **State/agency GIS portals**: Many state/international agencies maintain their own lidar portals with detailed metadata documentation.


In [ ]:
# Complete these fields if known, set to None if unknown

GEOID_COMPARE = None # e.g., "geoid12b"
EPOCH_COMPARE = "05/18/2005 - 05/27/2005"
HORIZ_CRS_COMPARE = None # e.g., "32611" 
VERT_CRS_COMPARE = None # e.g., "5703" 
COMPLETE_CRS_COMPARE = "4979" # e.g., "EPSG:3136+EPSG:5703"

GEOID_REFERENCE = "geoid12b"
EPOCH_REFERENCE = "05/27/2018 - 07/22/2018"
HORIZ_CRS_REFERENCE = None # e.g., "32611" 
VERT_CRS_REFERENCE = "5703" 
COMPLETE_CRS_REFERENCE = None # e.g., "EPSG:3136+EPSG:5703"

<h3><a id="setup2"></a>Continue setup</h3>

In [ ]:
import os, sys, pathlib
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Verify condacolab and install PDAL
    import condacolab
    condacolab.check()
    
    # Remove conflicting Python pin and install PDAL
    !rm -f /usr/local/conda-meta/pinned
    !mamba install -y -c conda-forge pdal python-pdal
    
    # Fix SQLite symlink - conda installs newer SQLite but old symlink remains
    # Dynamically find the installed SQLite version instead of hardcoding
    !sqlite_lib=$(ls /usr/local/lib/libsqlite3.so.3.* 2>/dev/null | head -1) && \
        if [ -n "$sqlite_lib" ]; then \
            sudo ln -sf "$sqlite_lib" /usr/local/lib/libsqlite3.so.0; \
            echo "Linked $sqlite_lib -> libsqlite3.so.0"; \
        fi
    
    # Fix numpy version conflict
    !{sys.executable} -m pip install -q "numpy<2.2"
    
    # Set PROJ environment
    os.environ['PROJ_LIB'] = '/usr/local/share/proj/'
    
    # Install topochange from GitHub
    print("\nInstalling topochange package...")
    !{sys.executable} -m pip install -q --no-cache-dir git+https://github.com/Cassandra-Brigham/topochange.git
    
    # Install additional packages not in topochange dependencies
    !{sys.executable} -m pip install -q small-gicp colormaps boto3
    
    # Verify PDAL via wrapper
    from topochange.pdal_wrapper import pdal, get_pdal_status
    status = get_pdal_status()
    print(f"\nEnvironment ready!")
    print(f"  PDAL version: {status['version']}")
    print(f"  PDAL mode: {status['mode']}")
else:
    print("Not running in Colab; skipping environment setup.")

In [ ]:
# Install visualization libraries
%pip install -q comm ipywidgets
%pip install -q ipyleaflet

# Fix pyproj PROJ path (Colab-specific, may need adjustment based on conda installation)
import os
import sys

# Set PROJ_LIB if needed (conda usually handles this automatically)
if IN_COLAB and not os.environ.get("PROJ_LIB"):
    from google.colab import output
    output.enable_custom_widget_manager()
    # Try common conda locations first
    possible_proj_paths = [
        "/opt/conda/share/proj",
        "/usr/share/proj", 
        "/usr/local/share/proj"
    ]
    for proj_path in possible_proj_paths:
        if os.path.isdir(proj_path):
            os.environ["PROJ_LIB"] = proj_path
            break

<h3><a id="import-libraries"></a>Import required libraries</h3>

This cell imports the necessary Python libraries for data handling and analysis, including custom functions from the provided scripts for differencing, variography, and stable area analysis.

In [ ]:
import numpy as np
import rasterio
from scipy.spatial.transform import Rotation

# Core classes
from topochange import (
    Raster,
    PointCloud,
    PointCloudPair,
    VariogramAnalysis,
    RasterDataHandler,
    StatisticalAnalysis,
    RegionalUncertaintyEstimator,
    VariogramModelSelector,
    FittedVariogramModel,
    MODEL_REGISTRY,
    CompositeVariogramModel,
)

# Interactive features (requires ipyleaflet)
from topochange import (
    TopoMapInteractor,
    StableAreaRasterizer,
    StableAreaAnalyzer,
)

# Geoid utilities
from topochange.geoid_utils import (
    select_geoid_grid,
    ensure_proj_grids_for_region,
    get_all_proj_data_dirs,
)

# Alignment
from topochange.alignment import LandscapeAligner, RegistrationConfig

SEED = 42

# Ensure PROJ geoid grids are available (Colab only)
if IN_COLAB:
    print("Setting up PROJ geoid grids for Colab...")
    print(f"PROJ data directories: {get_all_proj_data_dirs()}")
    ensure_proj_grids_for_region('us_noaa', verbose=True)
    print("\nPROJ geoid grids ready")

<h2><a id="data-access"></a>3. Data access, download and pre-processing</h2>

This section covers loading your compare (older) and reference (newer) topographic **point clouds**. 

Use this notebook if you have local `.las` or `.laz` point cloud files. The code will generate Digital Terrain Models (DTMs) and Digital Surface Models (DSMs) from your data and then align them. If you are starting with your own DEMs, go to [this notebook](). If you are not providing your own point clouds or DEMs and want to search for and download data, go to [this notebook](). 

<h3><a id="upload-laz"></a>Upload your point cloud files</h3>

In [ ]:
# Define point cloud file paths

compare_pc_path = get_data_path(COMPARE_PC_NAME)
reference_pc_path = get_data_path(REFERENCE_PC_NAME)

<h3><a id="extract_metadata"></a>Extract metadata from point cloud file</h3>

- Input format: use a point cloud in .las or .laz format.
- The path to the compare point cloud is already set in compare_pc_path.
- In the next cells, a PointCloud object (pc1) is created from compare_pc_path, its metadata is read from the file header/VLRs, and then printed for review.
- This step only inspects metadata (CRS, vertical datum/geoid, epoch, units, scales/offsets, bounds, classifications, returns, GPS time type); it does not modify the data.
- If key fields are missing or incorrect (e.g., compound CRS, horizontal/vertical CRS, geoid model, epoch), the following sections demonstrate how to update them using add_metadata before creating DEMs or performing transformations.


In [ ]:
# Create PointCloud object and read in metadata
pc1 = PointCloud(compare_pc_path)
pc1.from_file()

In [ ]:
pc1.print_metadata()

<h3><a id="update_metadata"></a>Update point cloud metadata</h3>

<h4><a id="update_compound"></a>Update compound Coordinate Reference System</h4>

In [ ]:
# Update compound CRS, using either a PROJ string, a WKT string, or an EPSG code.
# Skip this step if you are sure the compound CRS is correct

# If compound CRS is wrong or missing, can update metadata manually
  
if COMPLETE_CRS_COMPARE:
    pc1.add_metadata(compound_CRS=COMPLETE_CRS_COMPARE)
else:
    pass
print(f"Original compound CRS: {pc1.original_compound_crs}")  
print(f"Updated compound CRS: {pc1.current_compound_crs}")

<h4><a id="update_horiz"></a>Update horizontal Coordinate Reference System</h4>

In [ ]:
# Update horizontal CRS, using either a PROJ string, a WKT string, or an EPSG code.
# Skip this step if you are sure the horizontal CRS is correct.

if HORIZ_CRS_COMPARE:
    pc1.add_metadata(horizontal_CRS=HORIZ_CRS_COMPARE)
else:
    pass

print ("Original horizontal CRS:", pc1.original_horizontal_crs)
print("Original compound CRS:", pc1.original_compound_crs)
print("")
print(f"Updated horizontal CRS: {pc1.current_horizontal_crs}")
print(f"Updated compound CRS : {pc1.current_compound_crs}")

<h4><a id="update_vertical"></a>Update vertical Coordinate Reference System</h4>

In [ ]:
# Update vertical CRS, using either a PROJ string, a WKT string, or an EPSG code.

# Skip this step if you are sure the vertical CRS is correct.

if VERT_CRS_COMPARE:
    pc1.add_metadata(vertical_CRS=VERT_CRS_COMPARE)
else:
    pass

print ("Original vertical CRS:", pc1.original_vertical_crs)
print("Original compound CRS:", pc1.original_compound_crs)
print("")
print(f"Updated vertical CRS: {pc1.current_vertical_crs}")
print(f"Updated compound CRS : {pc1.current_compound_crs}")

<h4><a id="update_geoid"></a>Update geoid</h4>

In [ ]:
# If vertical coordinates are orthometric and geoid is wrong or missing, can update metadata manually
print(f"Original geoid model: \n {pc1.geoid_model}")
print(f"Original vertical CRS: \n {pc1.current_vertical_crs}")
print(f"Are the vertical coordinates orthometric?  \n {pc1.is_orthometric}")

In [ ]:
# Update geoid, using either a PROJ string, a WKT string, or an EPSG code.
if GEOID_COMPARE:
    geoid_grid, _ =select_geoid_grid(GEOID_COMPARE,verbose=True)
    
    # If the automatic selection is not the desired grid, you can specify the index of your desired grid in the list of matches. Uncomment the lines below and set choice to the desired index.
    #geoid_grid_2, _ =select_geoid_grid(geoid, choice=1)
    #print(f"Select the second grid: {geoid_grid_2}")
    
    pc1.add_metadata(geoid_model=geoid_grid)
else:
    pass

print(f"Geoid model: {pc1.geoid_model}")
print(f"Vertical CRS:{pc1.current_vertical_crs}")
print(f"Are the vertical coordinates orthometric? {pc1.is_orthometric}")

<h4><a id="update_epoch"></a>Update epoch</h4>

Accurate epoch metadata is critical when working with time-dependent coordinate reference systems and transformations.

- Many modern CRSs (e.g., ITRF, NAD83(2011), WGS84 realizations) are defined at a specific epoch and use velocities to propagate coordinates through time. Without the correct epoch, transformation pipelines cannot apply the right motion, leading to offsets.
- Horizontal positions can drift several centimeters per year due to tectonic motion; over a decade this can reach decimeter-level errors. Vertical biases can be similar or larger in areas with subsidence or uplift.
- When aligning point clouds to reference datasets collected at different times, mismatched epochs cause systematic misalignment that looks like a rigid shift or tilt rather than random noise.
- Velocity/deformation models and time-dependent Helmert transforms require an epoch to compute displacement; defaults (or missing epoch) often introduce hidden biases.
- Correct epoch improves repeatability, change detection, and metadata transparency, ensuring others can reproduce transformations and understand residuals.
- If data were collected over a range of dates, using the midpoint as the epoch is a practical, documented approximation for most transformations.

In [ ]:
# If epoch is wrong or missing, can update metadata
print(f"Original epoch: {pc1.epoch}")

In [ ]:
if EPOCH_COMPARE:
    # Update epoch, using either a decimal year, a single date, or a date range (the epoch will be set to the midpoint of the date range)
    pc1.add_metadata(epoch=EPOCH_COMPARE)
else:
    pass

print(f"Updated epoch: {pc1.epoch}")

In [ ]:
# Final check of all metadata
print("Current compound CRS:", pc1.current_compound_crs)
print("Current horizontal CRS:", pc1.current_horizontal_crs)
print("Current vertical CRS:", pc1.current_vertical_crs)
print("Current geoid model:", pc1.geoid_model)
print("Current epoch:", pc1.epoch)

<h3><a id="import_ref_cloud"></a>Import reference point cloud</h3>

In [ ]:
# Create PointCloud object and read in metadata
pc2 = PointCloud(reference_pc_path)
pc2.from_file()

In [ ]:
pc2.print_metadata()

In [ ]:
pc2.add_metadata(
    horizontal_CRS=HORIZ_CRS_REFERENCE if HORIZ_CRS_REFERENCE else None,
    vertical_CRS=VERT_CRS_REFERENCE if VERT_CRS_REFERENCE else None,
    compound_CRS=COMPLETE_CRS_REFERENCE if COMPLETE_CRS_REFERENCE else None,
    geoid_model=GEOID_REFERENCE if GEOID_REFERENCE else None,
    epoch=EPOCH_REFERENCE if EPOCH_REFERENCE else None,
                 )

In [ ]:
print("Current compound CRS:", pc2.current_compound_crs)
print("Current horizontal CRS:", pc2.current_horizontal_crs)
print("Current vertical CRS:", pc2.current_vertical_crs)
print("Current geoid model:", pc2.geoid_model)
print("Current epoch:", pc2.epoch)


<h3><a id="CRS_transformation"></a>Coordinate Reference System transformation</h3>

Before differencing, both datasets must share a common reference frame: not just the same map projection, but also the same vertical datum, geoid model, and coordinate epoch. Mismatches in any of these components introduce systematic errors that look like real elevation change.

| Component | What it defines | Error if mismatched |
|-----------|-----------------|---------------------|
| **Horizontal CRS** | Map projection and geodetic datum (e.g., UTM Zone 18N, NAD83) | Lateral shifts causing aspect-correlated vertical errors |
| **Vertical datum** | Height reference surface (ellipsoidal vs. orthometric) | Tens of meters if ellipsoid/orthometric confused |
| **Geoid model** | Equipotential surface relating ellipsoid to orthometric heights (e.g., GEOID12B, GEOID18) | 10–20 cm systematic offset |
| **Epoch** | Time of coordinate realization | cm/year drift from tectonic motion |

**The transformation pipeline:**

The `RasterPair.transform_raster1_to_match_raster2()` method applies transformations in a specific order:

1. **Epoch transformation** → Propagates coordinates through time using velocity models to account for tectonic plate motion. Critical when surveys span multiple years.

2. **Horizontal reprojection** → Converts between map projections and geodetic datums. Uses interpolation (bilinear recommended) to resample the grid.

3. **Vertical datum transformation** → Converts between ellipsoidal and orthometric heights, or between different geoid models. Applied as a Z-value adjustment.

4. **Grid alignment** → Resamples to match the target raster's exact pixel grid for cell-by-cell differencing.


**Checking for mismatches:**

Use `RasterPair.check_all_match()` or `PointCloudPair.check_all_match()` to identify which transformations are needed:
```python
comparison = raster_pair.check_all_match()
print(comparison['transformations_needed'])  # e.g., ['epoch', 'horizontal_crs', 'geoid']
```

**Common pitfalls:**

- Metadata may be incorrect or missing. Always verify CRS information against acquisition reports
- NAVD88 heights referenced to different geoid models (GEOID09, GEOID12B, GEOID18) are *not* directly comparable
- Legacy datasets often lack epoch information; use the acquisition date midpoint as a practical approximation

In [ ]:
# Create PointCloudPair
pc_pair = PointCloudPair(pc1, pc2)

In [ ]:
# Check what needs transformation
pc_pair.print_comparison()


In [ ]:
# Transform CRS to match reference
# Skip epoch transformation for faster processing if needed
transformed = pc_pair.transform_compare_to_match_reference(
    skip_epoch=True,  # Set to False for full epoch alignment
    verbose=True,
)

# Now compute overlap (after CRS transform so polygons are in same reference frame)
overlap_result = pc_pair.compute_overlap_polygon(use_transformed=True)
print(f"\nOverlap area: {overlap_result['overlap_area']:,.0f} mÂ²")
print(f"PC1 overlap: {overlap_result['overlap_fraction_pc1']*100:.1f}%")
print(f"PC2 overlap: {overlap_result['overlap_fraction_pc2']*100:.1f}%")

# Crop both clouds to overlap area (Area A)
pc1_cropped, pc2_cropped = pc_pair.crop_to_overlap(
    interior_buffer=10.0,  # Compare is 10m smaller on all edges
    verbose=True,
    overwrite=True,
)
print(f"\nPC1 cropped: {pc1_cropped.total_points:,} points")
print(f"PC2 cropped: {pc2_cropped.total_points:,} points")

<h3><a id="alignment"></a>Point cloud alignment</h3>

Before differencing, the two point clouds must be spatially aligned. Even small horizontal offsets between surveys produce apparent vertical changes that correlate with topographic aspect. Co-registration corrects these offsets by finding the optimal transformation that minimizes differences between overlapping stable terrain.

**Registration methods:**

The `LandscapeAligner` is built off of [small-gicp](https://github.com/koide3/small_gicp) and implements several Iterative Closest Point (ICP) variants:

| Method | Description | Best for |
|--------|-------------|----------|
| `icp` | Classic point-to-point ICP | Simple, fast alignment |
| `plane_icp` | Point-to-plane ICP | Smoother surfaces |
| `gicp` | Generalized ICP with covariance | Robust to noise, preferred for local use |
| `vgicp` | Voxelized GICP (GPU-accelerated) | Large point clouds, preferred for Colab use |

**Key parameters:**

- **`max_correspondence_distance`**: Maximum distance (meters) to consider point pairs as correspondences. Too small misses valid pairs; too large includes erroneous matches. Start with ~1 m for typical lidar.

- **`crop_dimensions`**: Cropping to a smaller region (e.g., 200×200 m) speeds computation and focuses alignment on a well-characterized area. The resulting transformation is then applied to the full dataset.

- **`point_filter="ground"`**: Using only ground-classified points avoids alignment errors from vegetation differences between surveys. If you want more classes than ground, you can provide a list of the class numbers you want to include (e.g. [2,6] for ground points and buildings.)


**Interpreting results:**

- **Fitness score**: Fraction of source points with valid correspondences (0–1). Values >0.5 typically indicate good alignment.
- **RMSE**: Root-mean-square error of point-to-point distances after alignment. Lower is better; values <0.1 m indicate excellent registration.
- **Transformation matrix**: The 4×4 matrix encoding the translation (and rotation if enabled) applied to align source → target.

After successful alignment, apply the transformation to your source point cloud before proceeding to DEM generation and differencing.

In [ ]:
if IN_COLAB:
    config = RegistrationConfig(
        point_filter="ground",
        max_correspondence_distance=1.0,
        crop_dimensions=(200, 200),
        method="vgicp",
    )
else:
    config = RegistrationConfig(
        point_filter="ground",
        max_correspondence_distance=1.0,
        method="gicp",
    )

aligner = LandscapeAligner(config)
result = aligner.align(source=pc1_cropped, target=pc2_cropped)

# # If you want to customize the registration parameters, you can create a RegistrationConfig object with your desired settings.
# # Uncomment and modify parameters as needed; defaults and parameter explanations shown below.

# config = RegistrationConfig(
#     # General parameters
#     method="gicp",  # str: "icp", "plane_icp", "gicp", "vgicp"
#     max_correspondence_distance=1.0,  # float or None: distance threshold (meters); None = auto-compute
#     max_iterations=50,  # int: maximum iterations for registration
    
#     # Centering and cropping
#     center_to_origin=True,  # bool: center both clouds to (0,0,0) before registration
#     crop_dimensions=None,  # tuple(float, float) or None: (x, y) crop rectangle in meters, centered on origin, or None to skip cropping
    
#     # Downsampling
#     downsample=False,  # bool: voxel grid downsampling using PDAL's filters.voxelcenternearestneighbor filter.
#     voxel_size=None,  # float or None: voxel size in meters; None = auto-compute when downsample=True, based on target_points
#     target_points=100000,  # int: target number of points after downsampling. if voxel_size is set, this is ignored.
    
#     # Coarse alignment
#     perform_coarse_alignment=True,  # bool: perform initial coarse alignment. If False, assumes clouds are roughly aligned already.
#     use_ground_plane_constraint=True,  # bool: If use_ground_plane_constraint=True (default for landscapes), only translation is applied, no rotation. This is appropriate for topographic data where both surveys should have the same "up" direction.
    
#     # Point filtering
#     point_filter="ground",  # str: "ground", "all", or "custom" (uses classification_filter)
#     use_ground_filter=False,  # bool: apply SMRF filter to classify ground (for unclassified data)
#     ground_filter_params={ # dict or None: SMRF parameters, e.g., {"cell": 1.0, "slope": 0.15, ...}
#         "cell": 1.0,       # Cell size (meters) for the grid used in morphological operations. Smaller values = finer detail but slower processing.
#         "scalar": 1.25,    # Elevation scalar for the progressive morphological filter. Controls how aggressively non-ground points are identified.
#         "slope": 0.15,     # Slope threshold (rise/run). Points with local slope exceeding this are candidates for non-ground classification.
#         "threshold": 0.5,  # Elevation threshold (meters). Maximum vertical distance a point can be from the ground surface to be classified as ground.
#         "window": 18.0,    # Maximum window size (meters) for morphological operations. Larger values handle bigger features (buildings, trees) but may smooth terrain.
#     }
    
#     # Outlier removal
#     outlier_removal=True,  # bool: remove statistical outliers before alignment
#     outlier_k_neighbors=20,  # int: number of neighbors for outlier detection
#     outlier_std_multiplier=2.0,  # float: standard deviations above mean to flag as outlier
    
#     # Classification filter (when point_filter="custom") ---
#     classification_filter=None,  # list[int] or None. E.g., [2] for ground, [2, 8] for ground + model key
    
#     # Validation thresholds
#     min_fitness_score=0.3,  # float: minimum acceptable fitness score (0-1). Fitness is the fraction of source points that found a valid correspondence in the target point cloud after alignment.
#     max_rmse=None,  # float or None: maximum acceptable RMSE; None: defaults to 5m
    
#     # Auto-retry on failure
#     enable_auto_retry=True,  # bool: retry with relaxed parameters if alignment fails
#     max_retries=3,  # int: maximum number of retry attempts
#     retry_strategies=[ # list[str]: strategies to try
#         "increase_correspondence", # x1.5 larger correspondence distance
#         "change_method",# on run 1 switch to vgicp if not already using it, on run 2 switch to icp
#         "adjust_filtering"], # relax outlier removal (outlier_std_multiplier Ã— 1.5) , use more points
# )

# aligner = LandscapeAligner(config)
# result = aligner.align(source=pc1_cropped, target=pc2_cropped)

In [ ]:
print(f"RMSE: {result.rmse:.4f} m")
print(f"Fitness: {result.fitness:.2%}")
print(f"Converged: {result.converged}")
print(f"Iterations: {result.iterations}")
print(f"Method: {result.method_used}")
print(f"\nTransformation matrix:\n{result.transformation}")
print(f"\nCentroid used: {result.centroid}")

# Extract rotation (3x3) and translation (3x1) from 4x4 transformation
rotation = result.transformation[:3, :3]
translation = result.transformation[:3, 3]

print(f"Translation (x, y, z): {translation}")
print(f"Rotation matrix:\n{rotation}")

# Convert rotation matrix to Euler angles (in degrees)
r = Rotation.from_matrix(rotation)
euler_angles = r.as_euler('xyz', degrees=True)
print(f"Rotation (roll, pitch, yaw) in degrees: {euler_angles}")

<h3><a id="differencing"></a>2D DEM differencing</h3>

Vertical differencing (also called raster subtraction or DoD—DEM of Difference) computes pixel-by-pixel elevation change between two co-registered DEMs. This is the fundamental operation for quantifying topographic change, producing a raster where each cell contains Δz = z₂ − z₁.

**Comparing processing scenarios:**

This example computes differences under three scenarios to illustrate how coordinate transformations and alignment affect results:

| Scenario | DEM Sources | What it tests |
|----------|-------------|---------------|
| **Horizontal-only** | Raw DSMs (no vertical transformation, same horizontal CRS) | Baseline; may include datum offsets |
| **Transformed** | After full 4D transformation (horizontal + vertical + epoch) | Effect of CRS transformation |
| **Aligned** | After transformation + ICP co-registration | Best-case scenario with geometric correction |

**Key parameters:**

- **`dem1`, `dem2`**: Specify which DEM products to difference. Options include `"dsm"`, `"dtm"`, `"dsm_transformed"`, `"dsm_transformed_aligned"`, etc.

- **`skip_epoch`**: If `True`, skips time-dependent coordinate transformations (e.g., plate motion corrections). Use to speed up process, when both datasets are already in the same epoch or when epoch differences are negligible.

**Interpreting the statistics:**

- **Mean/Median**: Non-zero values indicate systematic vertical bias. The median is more robust to outliers from real change or errors.

- **Standard deviation**: Quantifies the spread of elevation differences. High σ may indicate alignment issues, significant real change, or error sources like vegetation differences.

**What to look for in the difference maps:**

- *Aspect-correlated patterns* (positive on one slope aspect, negative on the opposite) → horizontal misalignment remains
- *Uniform offset across the scene* → vertical datum or calibration bias
- *Linear banding perpendicular to flight direction* → flight line errors ([Shan et al., 2007](https://doi.org/10.1201/9781420051438))
- *Localized clusters of change* → real geomorphic change or point misclassification

Comparing statistics across scenarios helps diagnose error sources: if alignment substantially reduces standard deviation, horizontal offsets were a dominant error. If the median shifts after transformation, vertical datum differences were present.

In [ ]:
output_dir = get_data_path("dem_output")
os.makedirs(output_dir, exist_ok=True)


# Scenario 1: Horizontal-only (NOT aligned)
results_horizontal_only = pc_pair.compute_2d_difference(
    dem1="dsm",
    dem2="dsm",
    overwrite=True,
    diff_output_path=os.path.join(output_dir, "diff_horizontal_only.tif"),
    verbose=True,
)

raster_pair_horizontal_only = results_horizontal_only['raster_pair']

# Scenario 2: Fully transformed (NOT aligned)
results_transformed = pc_pair.compute_2d_difference(
    dem1="dsm_transformed",
    dem2="dsm",
    skip_epoch=True,  # if you want to skip epoch
    overwrite=True,
    diff_output_path=os.path.join(output_dir, "diff_transformed.tif"),
    verbose=True,
)

raster_pair_transformed = results_transformed['raster_pair']

# Scenario 3: Fully transformed + ICP aligned
results_aligned = pc_pair.compute_2d_difference(
    dem1="dsm_transformed_aligned",
    dem2="dsm",
    skip_epoch=True,  # if you want to skip epoch
    overwrite=True,
    diff_output_path=os.path.join(output_dir, "diff_aligned.tif"),
    verbose=True,
)

raster_pair_aligned = results_aligned['raster_pair']

print(f"\n--- 2D Difference Statistics (Aligned) ---")
print(f"Mean: {results_aligned['stats']['mean']:.4f} m")
print(f"Median: {results_aligned['stats']['median']:.4f} m")
print(f"Std: {results_aligned['stats']['std']:.4f} m")

print(f"\n--- 2D Difference Statistics (Transformed) ---")
print(f"Mean: {results_transformed['stats']['mean']:.4f} m")
print(f"Median: {results_transformed['stats']['median']:.4f} m")
print(f"Std: {results_transformed['stats']['std']:.4f} m")

print(f"\n--- 2D Difference Statistics (Unaligned) ---")
print(f"Mean: {results_horizontal_only['stats']['mean']:.4f} m")
print(f"Median: {results_horizontal_only['stats']['median']:.4f} m")
print(f"Std: {results_horizontal_only['stats']['std']:.4f} m")

# Plot the difference raster
fig = raster_pair_aligned.plot_difference()

fig = raster_pair_transformed.plot_difference()
    
fig = raster_pair_horizontal_only.plot_difference()

In [ ]:
pair = raster_pair_aligned
results = results_aligned

<h2><a id="visualization"></a>4. Visualization and Derived Rasters</h2>

This section focuses on visualizing the results and creating derived topographic products like hillshades and slope maps, which are useful for interpreting the observed changes.

<h3><a id="plot-dems"></a>Plot the DEMs and derived rasters</h3>

In [ ]:
# Generate derivatives for both rasters
hillshade1, hillshade2 = pair.generate_derivative("hillshade")
slope1, slope2 = pair.generate_derivative("slope")
aspect1, aspect2 = pair.generate_derivative("aspect")
roughness1, roughness2 = pair.generate_derivative("roughness")

# Plot side-by-side with automatic derivative generation
fig, axes = pair.plot_pair(derivative='dem')        # Original DEMs
fig, axes = pair.plot_pair(derivative='hillshade')  # Hillshades
fig, axes = pair.plot_pair(derivative='slope')      # Slopes
fig, axes = pair.plot_pair(derivative='aspect')     # Aspects
fig, axes = pair.plot_pair(derivative='roughness')  # Roughness

# Customize hillshade parameters
fig, axes = pair.plot_pair(
    derivative='hillshade',
    azimuth=270,    # West-facing light
    altitude=30,    # Low sun angle
    figsize=(14, 6)
)

In [ ]:
pair.plot_difference()

<h2><a id="error_analysis"></a>5. Error analysis</h2>

<h3><a id="theoretical_framework"></a>Theoretical framework</h3>

The error analysis framework implemented here follows the geostatistical approach described by [Rolstad et al., 2009](https://doi.org/10.3189/002214309789470950) and [Hugonnet et al., 2022](https://doi.org/10.1109/JSTARS.2022.3188922), adapted for lidar-derived topographic differencing.

Topographic differencing errors can be decomposed into three components:

1. **Systematic vertical bias** (μ): A constant offset between datasets, typically estimated as the median of stable-area differences to reduce outlier influence

2. **Spatially correlated random error**: Errors that exhibit spatial structure due to:
   - Point misclassification (meter to decameter scale)
   - Geometric distortion on slopes (decameter scale)
   - Horizontal alignment errors (hundred-meter scale)
   - Flight line striping (hundred-meter to kilometer scale)
   - Vertical datum inconsistencies (kilometer scale)

3. **Uncorrelated random noise** (nugget): High-frequency noise from sensor sensitivity, environmental conditions, and surface reflectivity ([Glennie, 2007](https://doi.org/10.1515/jag.2007.017))

The semivariogram γ(h) quantifies how the variance of differences increases with separation distance h (Matheron, 1965; [Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277)):

$$\gamma(h) = \frac{1}{2N(h)} \sum_{i=1}^{N(h)} [\Delta_z(x_i) - \Delta_z(x_i + h)]^2$$

Key variogram parameters:
- **Nugget (c₀)**: Discontinuity at origin representing uncorrelated noise
- **Sill (c)**: Plateau value representing total variance
- **Range (a)**: Distance at which spatial correlation decays to negligible levels

For lidar data, **nested variograms** with multiple components capture error structures at different scales. Each spatial scale (short, medium, long range) contributes a portion of the total error variance.

<h3><a id="define_stable_areas"></a>Define stable areas</h3>


Stable areas are regions where **no topographic change is expected** between surveys. They serve as control zones for error calibration and the elevation differences within these areas represent pure error, allowing us to characterize the spatial structure of uncertainty.

<h4><a id="size_requirements"></a>Size requirements</h4>

The stable area must be large enough to reliably estimate the empirical variogram. Key constraints include (Journel & Huijbregts, 1978; [Oliver & Webster, 2015](https://doi.org/10.1007/978-3-319-15865-5)):

<h5><a id="min_pairs"></a>1. Minimum pairs per lag bin</h5>
Each lag bin in the variogram requires **at least 30 pairs**, with 50+ pairs recommended. Fewer than 20 pairs per bin produces unreliable estimates with high variance.

<h5><a id="var_coverage"></a>2. Variogram coverage rule</h5>
The variogram should span **less than half the domain size** to avoid pairing samples from opposite edges (Journel & Huijbregts, 1978). This means that if you expect correlation ranges up to 500 m, your stable area should have at least a 1 km extent in its longest dimension.

<h5><a id="corr_scales"></a>3. Capturing all correlation scales</h5> 
The largest correlation scale has the greatest impact on uncertainty estimates ([Rolstad et al., 2009](https://doi.org/10.3189/002214309789470950)). Typical lidar error correlation ranges span:

- **Short-range**: 10–100 m (point classification, geometric distortion)
- **Mid-range**: 100–1000 m (horizontal alignment, flight lines)
- **Long-range**: >1 km (vertical datum, geoid errors)

Your stable areas should extend far enough to capture all relevant scales.

<h4><a id="good_stable_area"></a>What makes a good stable area?</h4> 

**Ideal choices**:
- Roads and parking lots (paved, unvegetated)
- Bedrock outcrops (geologically stable)
- Flat, undisturbed terrain
- Areas with similar terrain properties (slope, roughness) to your areas of interest

**Areas to avoid**:
- Vegetation change zones (leaf-on vs. leaf-off)
- Construction or development areas
- Water bodies (variable water levels)
- Very steep slopes (higher geometric distortion)
- DEM boundary edges (edge effects)
- Areas with real expected change

<h4><a id="heteroscedasticity_consideration"></a>Heteroscedasticity consideration</h4>

Errors can vary with terrain properties. [Hugonnet et al., 2022](https://doi.org/10.1109/JSTARS.2022.3188922) explored how error variance changes with parameters such as slope and roughness. Ideally, stable areas should have similar terrain characteristics to your areas of interest, or you should account for this heteroscedasticity in the analysis. [xDEM](https://xdem.readthedocs.io/en/stable/uncertainty.html#heteroscedasticity) offers tools to model raster heteroscedasticity.



**Use the interactive map below to draw polygons over areas you consider stable.**

**Instructions:**
1. Use the polygon tool to draw areas of no expected change
2. Select "Stable" from the layer dropdown when drawing
3. Aim for areas totaling at least 1 km² if you expect long-range correlations
4. Run the cells below after drawing your polygons

In [ ]:
diff = results.get("difference_raster")

In [ ]:
out_folder_poly = Path.joinpath(BASE_DATA_DIR,"polygons/")
os.makedirs(out_folder_poly, exist_ok=True)


interactor = TopoMapInteractor(
    topo_diff_path=diff.filename,
    hillshade_path=hillshade1.filename,
    output_dir=out_folder_poly,
    overlay_dpi=600,
    overlay_vmin=-20,
    overlay_vmax=20,
)

interactor.map

In [ ]:
interactor.stable_geoms

In [ ]:
interactor.unstable_geoms

<h3><a id="descriptive_stats"></a>Descriptive statistics</h3>

Before fitting a variogram, it is essential to examine the statistical characteristics of the stable area(s). Descriptive statistics provide a first-order assessment of the differencing errors and help evaluate whether the data meet the assumptions required for geostatistical analysis.

**Key statistics to examine:**

- **Mean and Median**: The median of the elevation differences in stable areas estimates the systematic vertical bias between the two DEMs. A non-zero median indicates a consistent offset that should be removed before variogram analysis. The mean is more sensitive to outliers, so comparing mean and median helps identify skewness in the distribution.

- **Standard Deviation and Variance**: These quantify the overall spread of elevation differences. High variance may indicate significant error sources or residual real change in areas assumed to be stable.

- **Skewness and Kurtosis**: Departures from normality can signal issues. Positive skewness might indicate unremoved vegetation or construction, while heavy tails (high kurtosis) could reflect outliers from misclassification or edge effects.

- **Percentiles (0.5%, 99.5%)**: Examining extreme percentiles helps identify outliers that may need to be filtered before variogram estimation.

**Assessing stationarity across stable areas:**

A fundamental assumption in geostatistics is *stationarity* — that the mean and variance of the error field are constant across the study area, and that spatial covariance depends only on separation distance, not absolute location ([Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277)). Examining the statistics of each stable area *separately* provides a practical check:

1. **Compare means/medians across areas**: Substantially different median values may indicate spatially varying bias (e.g., from flight line effects or tilted datums), violating the assumption that a single variogram model applies everywhere.

2. **Compare variances across areas**: If one stable area has much higher variance than another, the error characteristics may depend on location or terrain properties. This *heteroscedasticity* suggests that a single variogram may not adequately represent error structure across the entire scene ([Hugonnet et al., 2022](https://doi.org/10.1109/JSTARS.2022.3188922)).

3. **Look for systematic patterns**: If stable areas on one side of the scene consistently show positive differences while those on the other side show negative differences, a regional trend or tilt may be present that should be modeled separately.

The table and histogram below summarize the distribution of elevation differences in your stable area(s). If you have defined multiple stable polygons, compare their individual statistics to assess spatial consistency.

In [ ]:
stable_polys, _ = interactor.export_geodataframes()

# one combined mask
rasterizer_stable = StableAreaRasterizer(interactor.topo_diff.path, stable_polys, nodata=-9999)
analyzer_stable = StableAreaAnalyzer(rasterizer_stable)

# Combined-area stats
df_all_stable_polys = analyzer_stable.stats_all(Path.joinpath(BASE_DATA_DIR,"polygons/combined_stable.tif"))

# Per-area stats
df_each_stable_poly = analyzer_stable.stats_each(Path.joinpath(BASE_DATA_DIR,"polygons/each_stable/"))


In [ ]:
df_all_stable_polys

In [ ]:
df_each_stable_poly

<h3><a id="estimate-error"></a>Estimate systematic error</h3>


Systematic error (vertical bias) represents a constant offset between the two DEMs that affects all elevation differences uniformly. This bias can arise from instrument calibration errors, incorrect atmospheric corrections, GNSS/IMU misalignments, or inconsistencies in vertical coordinate reference systems such as mismatched geoid models ([Glennie, 2007](https://doi.org/10.1515/jag.2007.017); [Habib et al., 2009](https://doi.org/10.14358/PERS.75.10.1159)). Geoid errors typically produce shifts of 10–20 cm, while confusion between ellipsoidal and orthometric heights can cause offsets of tens of meters.

**Why use the median?**

We estimate vertical bias as the **median** of elevation differences in stable areas rather than the mean. The median is more robust to outliers(isolated large values from real change, misclassification, or edge effects—ensuring the bias estimate reflects the typical offset rather than being skewed by anomalous values) (Brigham et al.)

**Interpreting the bias:**

- A non-zero median indicates systematic offset that should be removed before variogram analysis
- The bootstrap uncertainty quantifies confidence in the bias estimate
- After bias removal, the distribution should be approximately centered on zero

**Important considerations:**

- If you observe widespread real change (e.g., regional uplift from an earthquake or isostatic rebound), this signal will be absorbed into the bias estimate and must be accounted for separately
- For DSM differencing, vegetation changes (leaf-on vs. leaf-off) can bias the median; consider estimating bias from DTM results instead
- Very large biases (>1 m) may indicate datum errors that should be investigated before proceeding

In [ ]:

# Load the stable area raster (this is the masked difference raster)
stable_area_path = Path.joinpath(BASE_DATA_DIR, "polygons/combined_stable.tif")
stable_area = Raster.from_file(stable_area_path)

# Get the median from the stable area
# Read the data directly with rasterio to get valid values

with rasterio.open(stable_area_path) as src:
    data = src.read(1)
    nodata = src.nodata
    # Mask nodata values
    if nodata is not None:
        valid_data = data[data != nodata]
    else:
        valid_data = data[np.isfinite(data)]
    
    diff_stable_median = np.median(valid_data)
    print(f"Median of stable area differences: {diff_stable_median:.4f} m")

# Set up paths and parameters
output_path = Path.joinpath(BASE_DATA_DIR, "polygons/combined_stable_bias_removed.tif")
unit = "m"
dem_resolution = 1.0

# Load raster data using RasterDataHandler
raster_data_handler = RasterDataHandler(stable_area_path, unit, dem_resolution)
raster_data_handler.load_raster()

# Get the data array
vert_diff_array = raster_data_handler.data_array

# Measure of vertical bias (median)
vertical_bias = np.median(vert_diff_array)
print(f"Vertical bias: {vertical_bias:.4f} m")

# Get uncertainty in the median value by bootstrap resampling
stats = StatisticalAnalysis(raster_data_handler)
median_uncertainty = stats.bootstrap_uncertainty_subsample(n_bootstrap=1000, subsample_proportion=0.1)
print(f"Median uncertainty (bootstrap): {median_uncertainty:.4f} m")

# Subtract the vertical bias from the stable area raster and save
raster_data_handler.subtract_value_from_raster(output_path, vertical_bias)
print(f"Saved bias-removed raster to: {output_path}")

# Create new RasterDataHandler for the modified raster
raster_bias_removed = RasterDataHandler(output_path, unit, dem_resolution)
raster_bias_removed.load_raster()

print(f"\nBias-removed stats:")
print(f"  Mean: {np.mean(raster_bias_removed.data_array):.4f} m")
print(f"  Median: {np.median(raster_bias_removed.data_array):.4f} m")
print(f"  Std: {np.std(raster_bias_removed.data_array):.4f} m")

In [ ]:
fig = stats.plot_data_stats()

<h3><a id="Variography"></a>Variography</h3>


Variography is the process of estimating and modeling the spatial covariance structure of your data. For topographic differencing, we analyze how elevation errors are correlated across space ([Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277); [Oliver & Webster, 2014](https://doi.org/10.1016/j.catena.2013.09.006)).

<h4><a id="semivariogram"></a>The semivariogram</h4>

The **semivariogram** γ(h) measures the average squared difference between values separated by distance h (Matheron, 1965):

$$\gamma(h) = \frac{1}{2N(h)} \sum_{i=1}^{N(h)} [z(x_i) - z(x_i + h)]^2$$

At small distances, nearby points tend to have similar errors (low semivariance). As distance increases, the correlation breaks down and semivariance increases until it reaches a plateau (the sill).

The variogram has several key features:

1. **Nugget (c₀)**: The y-intercept, representing measurement noise and microscale variability below the sampling resolution

2. **Sill (c)**: The plateau value representing total variance. When the variogram reaches the sill, points are no longer spatially correlated.

3. **Range (a)**: The distance at which the sill is reached. Beyond this distance, errors are independent.

<h4><a id="nested_variogram"></a>Nested variograms for multi-scale error</h4> 

Lidar differencing errors operate at multiple scales simultaneously. A **nested variogram** captures this:

$$\gamma(h) = c_0 + \sum_{i=1}^{n} c_i \cdot \text{model}_i(h, a_i)$$

where each component (i) has its own partial sill (cᵢ) and range (aᵢ). For example:
- Component 1 (range ~30 m): Point classification errors
- Component 2 (range ~500 m): Flight line alignment errors
- Component 3 (range ~2 km): Geoid/datum inconsistencies

<h4><a id="model_selection"></a>Model Selection</h4>

We fit nested spherical models and select the best using:
- **AIC (Akaike Information Criterion)**: Balances fit quality vs. model complexity
- **Cross-validation**: Tests predictive performance on held-out data

The spherical model is commonly used because it has a finite range, provides smooth transitions, and handles nesting well ([Webster & Oliver, 2007](https://doi.org/10.1002/9780470517277)).

<h4><a id="sampling"></a>Sampling for computational efficiency</h4>

Computing all pairwise distances for a large raster is computationally prohibitive. We use:
- **Stratified random sampling**: Divide the raster into sub-grids and sample uniformly
- **Multiple realizations**: Repeat sampling ~30 times to estimate confidence bounds ([Ortiz & Deutsch, 2002](https://doi.org/10.1023/A:1014412218427))
- **Numba JIT compilation**: Accelerates the pairwise calculations

In [ ]:
#Create variogram analysis instance based on modified raster
V = VariogramAnalysis(raster_bias_removed)

#Calculate a mean variogram with 75 bins from variograms made over 10 runs
V.calculate_mean_variogram_numba(area_side = 250, samples_per_area = 400, max_samples = 1000000000, bin_width = 30, max_n_bins = 3000, n_runs = 30, max_lag_multiplier = 0.5)

In [ ]:
BOUNDS = None

V.fit_best_spherical_model(sigma_type='std', bounds=BOUNDS, seed=SEED)

In [ ]:
fig = V.plot_best_spherical_model()

In [ ]:
# Multi-model fitting, compare spherical, exponential, gaussian, matern
print("=" * 70)
print("MULTI-MODEL VARIOGRAM FITTING")
print("=" * 70)

# Fit all candidate models and select best by AIC
best_model = V.fit_best_model_auto(
    model_types=['spherical', 'exponential', 'gaussian', 'matern'],
    max_components=2,
    include_nugget=True,
    criterion='aic',
    compute_cv=True,
    n_bootstrap=500,
    seed=SEED,
)

print(f"\nBest model: {'+'.join(best_model.composite_model.component_names)}")
print(f"AIC: {best_model.aic:.2f}")
print(f"BIC: {best_model.bic:.2f}")
print(f"CV-RMSE: {best_model.cv_rmse:.4f}" if best_model.cv_rmse else "CV-RMSE: N/A")

In [ ]:
# View all candidate models ranked by AIC
print(V.get_model_comparison_summary())

In [ ]:
# Visualize top 3 models against empirical variogram
import matplotlib.pyplot as plt

selector = V.model_selector
fig, ax = plt.subplots(figsize=(12, 7))

# Plot empirical variogram with error bars
ax.errorbar(V.lags, V.mean_variogram, yerr=V.err_variogram, 
            fmt='ko', label='Empirical', alpha=0.5, markersize=4)

# Plot top 3 models
h = np.linspace(0, V.lags.max(), 300)
colors = ['#e41a1c', '#377eb8', '#4daf4a']  # red, blue, green
sorted_models = sorted(selector.fitted_models, key=lambda m: m.aic)[:3]

for i, model in enumerate(sorted_models):
    name = "+".join(model.composite_model.component_names)
    if model.composite_model.include_nugget:
        name += "+nugget"
    marker = " ★" if model is selector.best_model else ""
    ax.plot(h, model.predict(h), color=colors[i], 
            label=f"{name} (AIC={model.aic:.1f}){marker}", linewidth=2)

ax.set_xlabel('Lag Distance (m)', fontsize=12)
ax.set_ylabel('Semivariance', fontsize=12)
ax.legend(loc='lower right')
ax.set_title('Model Comparison: Top 3 by AIC', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare AIC vs BIC model selection
# BIC penalizes complexity more heavily, favoring simpler models

best_by_aic = V.model_selector.select_best(criterion='aic')
best_by_bic = V.model_selector.select_best(criterion='bic')

print("Model Selection Comparison:")
print("-" * 50)
aic_name = "+".join(best_by_aic.composite_model.component_names)
bic_name = "+".join(best_by_bic.composite_model.component_names)
print(f"AIC selects: {aic_name} (AIC={best_by_aic.aic:.2f})")
print(f"BIC selects: {bic_name} (BIC={best_by_bic.bic:.2f})")

if aic_name != bic_name:
    print(f"\n→ BIC chose a simpler model (fewer parameters)")
    print(f"  Use AIC for better prediction, BIC for parsimony")
else:
    print(f"\n→ Both criteria agree on the same model")

<h3><a id="uncertainty_propagation"></a>Uncertainty propagation</h3>

Once we have a fitted variogram model, we can propagate uncertainty to any area of interest ([Rolstad et al., 2009](https://doi.org/10.3189/002214309789470950)). This is the key step that transforms error characterization into actionable uncertainty bounds.

For a spatially averaged elevation change over an area A, the regional variance σ²_A depends on the spatial covariance structure:

$$\sigma_A^2 = \frac{1}{A^2} \int_A \int_A [\sigma_{\Delta_z}^2 - \gamma(h)] \, dx \, dy$$

**Intuition**: If all points in your area are highly correlated (small area relative to the range), errors move together and uncertainty is high. If points are spread over a large area with mixed positive and negative errors that partially cancel, uncertainty decreases.

A crucial insight: **uncertainty decreases as the area of aggregation increases**, but at different rates depending on the variogram parameters:
- Features smaller than the correlation range: errors are correlated and don't cancel out (uncertainty ≈ √sill)
- Features much larger than the correlation range: errors average toward zero (uncertainty ≈ √(sill / n_effective))

With a nested variogram, each component contributes independently:

$$\sigma_{total}^2 = \sigma_{nugget}^2 + \sigma_{short}^2 + \sigma_{mid}^2 + \sigma_{long}^2$$

For irregularly shaped polygons, we use **Monte Carlo integration**: randomly sample N pairs of points within the polygon, compute covariances using the fitted variogram, and average to approximate the double integral (~10,000–25,000 pairs).

<h4><a id="features_of_interest"></a>Draw features of interest (unstable areas)</h4> 

Use the interactive map below to draw polygons around areas where you expect topographic change (e.g., landslides, construction sites, eroded channels, deposited sediment bars). The uncertainty will be calculated for each polygon.

In [ ]:
interactor.map

In [ ]:
interactor.unstable_geoms

In [ ]:
_, unstable_polys = interactor.export_geodataframes()

In [ ]:
uncertainties_per_feature_of_interest = []
for i, poly in enumerate(unstable_polys['geometry']):
    uncertainty_temp = RegionalUncertaintyEstimator(raster_bias_removed, V, area_of_interest=poly)
    uncertainty_temp.calc_mean_uncertainty(n_pairs=25_000, seed=SEED, sigma_func=None)
    uncertainties_per_feature_of_interest.append(uncertainty_temp)

total_mean_uncertainty_raster = uncertainties_per_feature_of_interest[0].total_mean_uncertainty_raster
total_mean_uncertainty_min_raster = uncertainties_per_feature_of_interest[0].total_mean_uncertainty_min_raster
total_mean_uncertainty_max_raster = uncertainties_per_feature_of_interest[0].total_mean_uncertainty_max_raster

mean_random_uncorrelated_raster = uncertainties_per_feature_of_interest[0].mean_random_uncorrelated_raster

total_mean_correlated_uncertainties_polygon = []
total_mean_correlated_uncertainties_polygon_min = []
total_mean_correlated_uncertainties_polygon_max = []

total_mean_uncorrelated_uncertainties_polygon = []

total_mean_uncertainties_polygon = []
total_mean_uncertainty_min_polygon = []
total_mean_uncertainty_max_polygon = []

for uncertainty in uncertainties_per_feature_of_interest:
    mean_correlated_uncertainty_polygon = uncertainty.total_mean_correlated_uncertainty_polygon
    mean_correlated_uncertainty_min_polygon = uncertainty.total_mean_correlated_uncertainty_min_polygon
    mean_correlated_uncertainty_max_polygon = uncertainty.total_mean_correlated_uncertainty_max_polygon
    
    mean_uncertainty_polygon = uncertainty.total_mean_uncertainty_polygon
    mean_uncertainty_min_polygon = uncertainty.total_mean_uncertainty_min_polygon
    mean_uncertainty_max_polygon = uncertainty.total_mean_uncertainty_max_polygon
    
    mean_random_uncorrelated = uncertainty.mean_random_uncorrelated
    
    total_mean_correlated_uncertainties_polygon.append(mean_correlated_uncertainty_polygon)
    total_mean_correlated_uncertainties_polygon_min.append(mean_correlated_uncertainty_min_polygon)
    total_mean_correlated_uncertainties_polygon_max.append(mean_correlated_uncertainty_max_polygon)
    
    total_mean_uncorrelated_uncertainties_polygon.append(mean_random_uncorrelated)
    
    total_mean_uncertainties_polygon.append(mean_uncertainty_polygon)
    total_mean_uncertainty_min_polygon.append(mean_uncertainty_min_polygon)
    total_mean_uncertainty_max_polygon.append(mean_uncertainty_max_polygon)


    

In [ ]:
# Summary of uncertainty estimates per polygon
print("=" * 70)
print("UNCERTAINTY SUMMARY PER FEATURE OF INTEREST")
print("=" * 70)

for i in range(len(uncertainties_per_feature_of_interest)):
    print(f"\nPolygon {i + 1}:")
    print(f"  Correlated uncertainty:   {total_mean_correlated_uncertainties_polygon[i]:.4f} m "
          f"[{total_mean_correlated_uncertainties_polygon_min[i]:.4f}, "
          f"{total_mean_correlated_uncertainties_polygon_max[i]:.4f}]")
    print(f"  Uncorrelated uncertainty: {total_mean_uncorrelated_uncertainties_polygon[i]:.4f} m")
    print(f"  Total uncertainty:        {total_mean_uncertainties_polygon[i]:.4f} m "
          f"[{total_mean_uncertainty_min_polygon[i]:.4f}, "
          f"{total_mean_uncertainty_max_polygon[i]:.4f}]")

print("\n" + "=" * 70)

---

<h2><a id="references"></a>References</h2>

- Albino, F., Smets, B., d'Oreye, N. & Kervyn, F. (2015). High‐resolution TanDEM‐X DEM: An accurate method to estimate lava flow volumes at Nyamulagira Volcano (D.R. Congo). *Journal of Geophysical Research: Solid Earth*, 120, 4189–4207. [https://doi.org/10.1002/2015JB011988](https://doi.org/10.1002/2015JB011988)

- Anderson, S.W. (2019). Uncertainty in quantitative analyses of topographic change: error propagation and the role of thresholding. *Earth Surface Processes and Landforms*, 44, 1015–1033. [https://doi.org/10.1002/esp.4551](https://doi.org/10.1002/esp.4551)

- Besl, P.J. & McKay, N.D. (1992). Method for registration of 3-D shapes. *Sensor Fusion IV: Control Paradigms and Data Structures*, SPIE, 586–606. [https://doi.org/10.1117/12.57955](https://doi.org/10.1117/12.57955)

- Brasington, J. & Smart, R.M.A. (2003). Close range digital photogrammetric analysis of experimental drainage basin evolution. *Earth Surface Processes and Landforms*, 28, 231–247. [https://doi.org/10.1002/esp.480](https://doi.org/10.1002/esp.480)

- Brasington, J., Langham, J. & Rumsby, B. (2003). Methodological sensitivity of morphometric estimates of coarse fluvial sediment transport. *Geomorphology*, 53, 299–316. [https://doi.org/10.1016/S0169-555X(02)00320-3](https://doi.org/10.1016/S0169-555X(02)00320-3)

- Dehecq, A., Gardner, A.S., Alexandrov, O., McMichael, S., Hugonnet, R., Shean, D. & Marty, M. (2020). Automated Processing of Declassified KH-9 Hexagon Satellite Images for Global Elevation Change Analysis Since the 1970s. *Frontiers in Earth Science*, 8, 566802. [https://doi.org/10.3389/feart.2020.566802](https://doi.org/10.3389/feart.2020.566802)

- Glennie, C.L., Hinojosa‐Corona, A., Nissen, E., Kusari, A., Oskin, M.E., Arrowsmith, J.R. & Borsa, A. (2014). Optimization of legacy lidar data sets for measuring near‐field earthquake displacements. *Geophysical Research Letters*, 41, 3494–3501. [https://doi.org/10.1002/2014GL059919](https://doi.org/10.1002/2014GL059919)

- Glennie, C. (2007). Rigorous 3D error analysis of kinematic scanning LIDAR systems. *Journal of Applied Geodesy*, 1, 147–157. [https://doi.org/10.1515/jag.2007.017](https://doi.org/10.1515/jag.2007.017)

- Heritage, G.L., Milan, D.J., Large, A.R.G. & Fuller, I.C. (2009). Influence of survey strategy and interpolation model on DEM quality. *Geomorphology*, 112, 334–344. [https://doi.org/10.1016/j.geomorph.2009.06.024](https://doi.org/10.1016/j.geomorph.2009.06.024)

- Hugonnet, R., Brun, F., Berthier, E., Dehecq, A., Mannerfelt, E.S., Eckert, N. & Farinotti, D. (2022). Uncertainty analysis of digital elevation models by spatial inference from stable terrain. *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*, 15, 6456–6472. [https://doi.org/10.1109/JSTARS.2022.3188922](https://doi.org/10.1109/JSTARS.2022.3188922)

- Izumida, A., Uchiyama, S. & Sugai, T. (2017). Application of UAV-SfM photogrammetry and aerial lidar to a disastrous flood: repeated topographic measurement of a midstream 2 river during a recovery. *Natural Hazards and Earth System Sciences*, 17, 1505–1519. [https://doi.org/10.5194/nhess-17-1505-2017](https://doi.org/10.5194/nhess-17-1505-2017)

- Journel, A.G. & Huijbregts, C.J. (1978). *Mining Geostatistics*. Academic Press.

- Lane, S.N., Westaway, R.M. & Hicks, D.M. (2003). Estimation of erosion and deposition volumes in a large, gravel‐bed, braided river using synoptic remote sensing. *Earth Surface Processes and Landforms*, 28, 249–271. [https://doi.org/10.1002/esp.483](https://doi.org/10.1002/esp.483)

- Langridge, R.M., Ries, W.F., Farrier, T., Barth, N.C., Khajavi, N. & De Pascale, G.P. (2014). Developing sub 5-m lidar DEMs for forested sections of the Alpine and Hope faults, South Island, New Zealand. *Geomorphology*, 226, 226–240. [https://doi.org/10.1016/j.geomorph.2014.08.007](https://doi.org/10.1016/j.geomorph.2014.08.007)

- Lucieer, A., de Jong, S.M. & Turner, D. (2014). Mapping landslide displacements using Structure from Motion (SfM) and image correlation of multi-temporal UAV photography. *Progress in Physical Geography*, 38, 97–116. [https://doi.org/10.1177/0309133313515293](https://doi.org/10.1177/0309133313515293)

- Matheron, G. (1965). *Les variables régionalisées et leur estimation*. Masson, Paris.

- Oliver, M.A. & Webster, R. (2014). A tutorial guide to geostatistics: Computing and modelling variograms and kriging. *Catena*, 113, 56–69. [https://doi.org/10.1016/j.catena.2013.09.006](https://doi.org/10.1016/j.catena.2013.09.006)

- Oliver, M.A. & Webster, R. (2015). *Basic Steps in Geostatistics: The Variogram and Kriging*. Springer. [https://doi.org/10.1007/978-3-319-15865-5](https://doi.org/10.1007/978-3-319-15865-5)

- Ortiz, J.M. & Deutsch, C.V. (2002). Calculation of uncertainty in the variogram. *Mathematical Geology*, 34(2), 169–183. [https://doi.org/10.1023/A:1014412218427](https://doi.org/10.1023/A:1014412218427)

- Passalacqua, P., Belmont, P., Staley, D.M. et al. (2015). Analyzing high resolution topography for advancing the understanding of mass and energy transfer through landscapes: A review. *Earth-Science Reviews*, 148, 174–193. [https://doi.org/10.1016/j.earscirev.2015.05.012](https://doi.org/10.1016/j.earscirev.2015.05.012)

- Rolstad, C., Haug, T. & Denby, B. (2009). Spatially integrated geodetic glacier mass balance and its uncertainty based on geostatistical analysis: application to the western Svartisen ice cap, Norway. *Journal of Glaciology*, 55(192), 666–680. [https://doi.org/10.3189/002214309789470950](https://doi.org/10.3189/002214309789470950)

- Schaffrath, K.R., Belmont, P. & Wheaton, J.M. (2015). Landscape-scale geomorphic change detection: Quantifying spatially variable uncertainty and circumventing legacy data issues. *Geomorphology*, 250, 334–348. [https://doi.org/10.1016/j.geomorph.2015.09.020](https://doi.org/10.1016/j.geomorph.2015.09.020)

- Scott, C., Arrowsmith, J.R., Nissen, E., Lajoie, L., Maruyama, T. & Chiba, T. (2018). The M7 2016 Kumamoto, Japan, Earthquake: 3-D Deformation Along the Fault and Within the Damage Zone Constrained From Differential Lidar Topography. *Journal of Geophysical Research: Solid Earth*, 123, 6138–6155. [https://doi.org/10.1029/2018JB015581](https://doi.org/10.1029/2018JB015581)

- Scott, C., Phan, M., Nandigam, V., Crosby, C. & Arrowsmith, J.R. (2021). Measuring change at Earth's surface: On-demand vertical and three-dimensional topographic differencing implemented in OpenTopography. *Geosphere*, 17, 1318–1332. [https://doi.org/10.1130/GES02259.1](https://doi.org/10.1130/GES02259.1)

- Webster, R. & Oliver, M.A. (2007). *Geostatistics for Environmental Scientists*, 2nd ed. John Wiley & Sons. [https://doi.org/10.1002/9780470517277](https://doi.org/10.1002/9780470517277)

- Wheaton, J.M., Brasington, J., Darby, S.E. & Sear, D.A. (2010). Accounting for uncertainty in DEMs from repeat topographic surveys: improved sediment budgets. *Earth Surface Processes and Landforms*, 35, 136–156. [https://doi.org/10.1002/esp.1886](https://doi.org/10.1002/esp.1886)

